# Предобработка признаков

В этом ноутбуке разберём **предобработку признаков** — обязательный этап перед обучением моделей. От качества и осмысленности предобработки часто зависит успех всего пайплайна.

**Цели ноутбука:**
- Понять, зачем нужны масштабирование и нормализация признаков и когда какой метод применять.
- Разобрать работу с категориальными признаками (кодирование).
- Рассмотреть простые стратегии обработки пропусков и выбросов.
- На практике применить `StandardScaler`, `MinMaxScaler`, `OneHotEncoder`, `SimpleImputer` из `sklearn`.

---

**Содержание:** Зачем предобработка → Масштабирование → Категориальные признаки → Пропуски и выбросы → Пайплайн в sklearn → Выводы.


## Зачем нужна предобработка

Многие алгоритмы машинного обучения **чувствительны к масштабу признаков**:
- **kNN**, **SVM**, **логистическая регрессия** опираются на расстояния или скалярные произведения; если один признак измерен в тысячах, а другой в долях, первый будет доминировать.
- **Градиентный спуск** сходится быстрее и стабильнее на отмасштабированных признаках.

Кроме того:
- модели обычно принимают **числовые** векторы — категориальные признаки нужно превращать в числа;
- в реальных данных встречаются **пропуски** (NaN) и **выбросы**, с которыми нужно определиться до обучения.

**Важно:** масштабирование (и любую подгонку параметров преобразований) делают **только по обучающей выборке**, затем те же параметры применяют к тесту, чтобы не было "утечки" информации из теста в обучение.


## Масштабирование признаков

**StandardScaler** (стандартизация): вычитаем среднее и делим на стандартное отклонение по каждому признаку. После преобразования каждый признак имеет нулевое среднее и единичную дисперсию. Подходит для большинства линейных и distance-based моделей.

**MinMaxScaler**: масштабируем каждый признак в заданный диапазон (по умолчанию [0, 1]) по формуле $x_{new} = (x - x_{min}) / (x_{max} - x_{min})$. Чувствителен к выбросам.

**RobustScaler**: использует медиану и межквартильный размах (IQR), поэтому устойчив к выбросам.

Ниже сравниваем влияние масштабирования на kNN.


In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Сгенерируем данные, где признаки на очень разных масштабах
X, y = make_classification(
    n_samples=600,
    n_features=5,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=42,
)

# Умножим первый признак на 1000, а второй сильно уменьшим
X[:, 0] *= 1000.0
X[:, 1] *= 0.001

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Без масштабирования
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train, y_train)
acc_raw = accuracy_score(y_test, knn_raw.predict(X_test))

# StandardScaler
scaler_std = StandardScaler()
X_train_std = scaler_std.fit_transform(X_train)
X_test_std = scaler_std.transform(X_test)
knn_std = KNeighborsClassifier(n_neighbors=5)
knn_std.fit(X_train_std, y_train)
acc_std = accuracy_score(y_test, knn_std.predict(X_test_std))

# MinMaxScaler
scaler_mm = MinMaxScaler()
X_train_mm = scaler_mm.fit_transform(X_train)
X_test_mm = scaler_mm.transform(X_test)
knn_mm = KNeighborsClassifier(n_neighbors=5)
knn_mm.fit(X_train_mm, y_train)
acc_mm = accuracy_score(y_test, knn_mm.predict(X_test_mm))

print("Точность kNN (k=5) на сильно разном масштабе признаков:")
print(f"  Без масштабирования: {acc_raw:.3f}")
print(f"  StandardScaler:      {acc_std:.3f}")
print(f"  MinMaxScaler:        {acc_mm:.3f}")

Точность kNN (k=5) на сильно разном масштабе признаков:
  Без масштабирования: 0.489
  StandardScaler:      0.928
  MinMaxScaler:        0.950


## Категориальные признаки

Если признак принимает конечное число значений (например, цвет: красный, зелёный, синий), его нужно **закодировать** в числа.

- **LabelEncoder**: каждому категориальному значению сопоставляется одно число (0, 1, 2, …). Удобно для **целевой переменной** (меток классов). Для признаков нежелательно: модель может решить, что "2" больше "1", хотя порядок мог быть произвольным.

- **OneHotEncoder**: для каждой категории создаётся отдельный бинарный признак (1, если объект принадлежит этой категории, иначе 0). Порядок не вносится, но число признаков растёт. В sklearn часто используют `ColumnTransformer` вместе с `OneHotEncoder`, чтобы применять разные преобразования к разным столбцам.

Пример: кодируем категориальный признак и объединяем с числовыми.


In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Имитация данных: числовые признаки + категориальный (город: A, B, C)
np.random.seed(42)
n = 100
X_num = np.random.randn(n, 2)
city = np.random.choice(["A", "B", "C"], size=n)
X_cat = city.reshape(-1, 1)

# ColumnTransformer: к числовым столбцам ничего не делаем, категориальный — OneHot
ct = ColumnTransformer(
    [("num", "passthrough", [0, 1]), ("cat", OneHotEncoder(sparse_output=False), [2])],
    remainder="drop",
)
# Таблица: столбцы 0,1 — числовые, столбец 2 — категориальный (город)
X_table = np.hstack([X_num, X_cat.astype(object)])
ct.fit(X_table)
X_encoded = ct.transform(X_table)
print("Форма до кодирования:", X_table.shape)
print("Форма после OneHot по столбцу города:", X_encoded.shape)
print("Пример строки (2 числа + 3 бинарных для A,B,C):", X_encoded[0])

Форма до кодирования: (100, 3)
Форма после OneHot по столбцу города: (100, 5)
Пример строки (2 числа + 3 бинарных для A,B,C): [0.4967141530112327 -0.13826430117118466 0.0 1.0 0.0]


In [3]:
import pandas as pd

# Визуализируем данные до и после кодирования
cols_before = ["num_0", "num_1", "city"]
df_before = pd.DataFrame(X_table, columns=cols_before)

# OneHotEncoder внутри ColumnTransformer создаёт новые признаки для A,B,C
cols_after = ["num_0", "num_1", "city_A", "city_B", "city_C"]
df_after = pd.DataFrame(X_encoded, columns=cols_after)

print("Первые 5 строк ДО кодирования:")
display(df_before.head())

print("Первые 5 строк ПОСЛЕ OneHot-кодирования:")
display(df_after.head())

Первые 5 строк ДО кодирования:


,num_0,num_1,city
0,0.496714,-0.138264,B
1,0.647689,1.52303,B
2,-0.234153,-0.234137,A
3,1.579213,0.767435,A
4,-0.469474,0.54256,A


Первые 5 строк ПОСЛЕ OneHot-кодирования:


,num_0,num_1,city_A,city_B,city_C
0,0.496714,-0.138264,0.0,1.0,0.0
1,0.647689,1.52303,0.0,1.0,0.0
2,-0.234153,-0.234137,1.0,0.0,0.0
3,1.579213,0.767435,1.0,0.0,0.0
4,-0.469474,0.54256,1.0,0.0,0.0


## Пропуски и выбросы

**Пропуски (NaN):**
- **SimpleImputer** в sklearn умеет заполнять пропуски стратегиями: среднее, медиана, наиболее частый класс (для категорий). Важно вызывать `fit` только на train, затем `transform` на train и test.
- Альтернативы: удалить объекты с пропусками или признаки с большим числом пропусков; для деревьев иногда допускают пропуски как отдельное значение.

**Выбросы:**
- Визуализация (ящики с усами, scatter) и предметное понимание: иногда выброс — ошибка измерения, иногда — редкое, но важное событие.
- Можно заменить экстремальные значения на перцентили, применить RobustScaler или удалить выбросы по правилу (например, за пределами 1.5·IQR от квартилей). В sklearn нет единого "выбросоудалятеля", но есть `SimpleImputer` с стратегией по перцентилям и методы вроде `EllipticEnvelope` для детекции.

Краткий пример заполнения пропусков медианой:


In [4]:
from sklearn.impute import SimpleImputer

X_with_missing = X_train.copy()
np.random.seed(42)
mask = np.random.rand(*X_train.shape) < 0.05  # 5% пропусков
X_with_missing[mask] = np.nan

imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X_with_missing)
print("Число пропусков до:", np.isnan(X_with_missing).sum())
print("Число пропусков после:", np.isnan(X_imputed).sum())
print("Медианы по столбцам (подставлены вместо NaN):", imputer.statistics_)

Число пропусков до: 108
Число пропусков после: 0
Медианы по столбцам (подставлены вместо NaN): [-2.19527173e+00  3.08118643e-04  1.58120983e-02  7.06315863e-03
  9.41370082e-01]


## Пайплайн в sklearn

Чтобы не забыть применить одни и те же шаги предобработки к тесту, удобно объединять преобразования и модель в **Pipeline**. Тогда и масштабирование, и импут, и модель обучаются одним вызовом `fit` на train; для предсказания на тесте вызывается `predict`, и все этапы применяются автоматически.


In [5]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])
pipe.fit(X_with_missing, y_train)
acc_pipe = accuracy_score(y_test, pipe.predict(X_test))
print("Точность пайплайна (Imputer → StandardScaler → kNN) на тесте:", f"{acc_pipe:.3f}")

Точность пайплайна (Imputer → StandardScaler → kNN) на тесте: 0.933


## Выводы

- Предобработка обязательна для distance-based и градиентных моделей: масштабирование (чаще всего StandardScaler) делают по обучающей выборке и тем же преобразованием обрабатывают тест.
- Категориальные признаки кодируют (OneHotEncoder для признаков, LabelEncoder для меток); при смешанных типах данных удобен ColumnTransformer.
- Пропуски заполняют стратегией (медиана, среднее, константа), выбросы обрабатывают с учётом предметной области; всё объединяют в Pipeline, чтобы избежать утечки и упростить код.
